# GNN Retrieval Training from Neo4j (`Chunk`-`Entity`)

Notebook строит обучающий пайплайн для retrieval на графе Neo4j:
- читает структуру `(:Chunk)-[:MENTIONS]->(:Entity)` и `(:Entity)-[:RELATED_TO]->(:Entity)`,
- формирует гетерогенный граф,
- обучает GraphSAGE для ранжирования `Entity` по `Chunk` (link prediction),
- считает `Recall@K`/`MRR`,
- записывает `gnn_embedding` обратно в Neo4j.


In [ ]:
# При необходимости раскомментируйте установку зависимостей
# %pip install neo4j pandas numpy scikit-learn torch torch-geometric


In [ ]:
import os
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from neo4j import GraphDatabase
from sklearn.feature_extraction.text import HashingVectorizer
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


In [ ]:
# Конфиг подключения
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'neo4j_password')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(query: str, params: dict | None = None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [r.data() for r in result]

schema_probe = run_query("""
MATCH (c:Chunk)
OPTIONAL MATCH (c)-[:MENTIONS]->(e:Entity)
RETURN count(DISTINCT c) AS chunks, count(DISTINCT e) AS entities, count(*) AS mention_rows
""")
schema_probe


In [ ]:
# Вытягиваем узлы/ребра с безопасными fallback полями (как в вашем скрине: entity_name, chunk_id и т.д.)
chunks = run_query("""
MATCH (c:Chunk)
RETURN
  c.chunk_id AS chunk_id,
  c.doc_id AS doc_id,
  coalesce(c.text, '') AS text
""")

entities = run_query("""
MATCH (e:Entity)
RETURN
  e.entity_id AS entity_id,
  coalesce(e.entity_name, e.normalized_value, e.value, '') AS entity_text
""")

mentions = run_query("""
MATCH (c:Chunk)-[:MENTIONS]->(e:Entity)
RETURN c.chunk_id AS chunk_id, e.entity_id AS entity_id
""")

related = run_query("""
MATCH (e1:Entity)-[:RELATED_TO]->(e2:Entity)
RETURN e1.entity_id AS src_entity_id, e2.entity_id AS dst_entity_id
""")

chunks_df = pd.DataFrame(chunks).dropna(subset=['chunk_id']).drop_duplicates('chunk_id')
entities_df = pd.DataFrame(entities).dropna(subset=['entity_id']).drop_duplicates('entity_id')
mentions_df = pd.DataFrame(mentions).dropna().drop_duplicates()
related_df = pd.DataFrame(related).dropna().drop_duplicates()

print('chunks:', len(chunks_df))
print('entities:', len(entities_df))
print('mentions:', len(mentions_df))
print('related_to:', len(related_df))
chunks_df.head(2), entities_df.head(2), mentions_df.head(2)


In [ ]:
# Фичи узлов через HashingVectorizer (без внешних моделей и скачиваний)
chunk_texts = chunks_df['text'].fillna('').astype(str).tolist()
entity_texts = entities_df['entity_text'].fillna('').astype(str).tolist()

vectorizer = HashingVectorizer(n_features=512, alternate_sign=False, norm='l2')
chunk_x = vectorizer.transform(chunk_texts).astype(np.float32).toarray()
entity_x = vectorizer.transform(entity_texts).astype(np.float32).toarray()

chunk2idx = {cid: i for i, cid in enumerate(chunks_df['chunk_id'].tolist())}
entity2idx = {eid: i for i, eid in enumerate(entities_df['entity_id'].tolist())}

mentions_df = mentions_df[mentions_df['chunk_id'].isin(chunk2idx) & mentions_df['entity_id'].isin(entity2idx)].copy()
related_df = related_df[related_df['src_entity_id'].isin(entity2idx) & related_df['dst_entity_id'].isin(entity2idx)].copy()

mention_src = torch.tensor([chunk2idx[x] for x in mentions_df['chunk_id']], dtype=torch.long)
mention_dst = torch.tensor([entity2idx[x] for x in mentions_df['entity_id']], dtype=torch.long)

rel_src = torch.tensor([entity2idx[x] for x in related_df['src_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)
rel_dst = torch.tensor([entity2idx[x] for x in related_df['dst_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)

data = HeteroData()
data['chunk'].x = torch.from_numpy(chunk_x)
data['entity'].x = torch.from_numpy(entity_x)

data['chunk', 'mentions', 'entity'].edge_index = torch.stack([mention_src, mention_dst], dim=0)
data['entity', 'mentioned_in', 'chunk'].edge_index = torch.stack([mention_dst, mention_src], dim=0)
data['entity', 'related_to', 'entity'].edge_index = torch.stack([rel_src, rel_dst], dim=0)
data['entity', 'related_to_rev', 'entity'].edge_index = torch.stack([rel_dst, rel_src], dim=0)

data


In [ ]:
# Train/Val/Test split по рёбрам MENTIONS
all_edges = data['chunk', 'mentions', 'entity'].edge_index.t().cpu().numpy()
perm = np.random.permutation(len(all_edges))
all_edges = all_edges[perm]

n = len(all_edges)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_edges = torch.tensor(all_edges[:n_train], dtype=torch.long)
val_edges = torch.tensor(all_edges[n_train:n_train+n_val], dtype=torch.long)
test_edges = torch.tensor(all_edges[n_train+n_val:], dtype=torch.long)

num_chunks = data['chunk'].x.shape[0]
num_entities = data['entity'].x.shape[0]

existing = set((int(c), int(e)) for c, e in all_edges.tolist())

def sample_negative(batch_edges: torch.Tensor, max_tries: int = 20) -> torch.Tensor:
    neg = batch_edges.clone()
    for i in range(len(neg)):
        c = int(neg[i, 0])
        for _ in range(max_tries):
            e = random.randrange(num_entities)
            if (c, e) not in existing:
                neg[i, 1] = e
                break
    return neg

len(train_edges), len(val_edges), len(test_edges)


In [ ]:
class RetrievalHeteroSAGE(nn.Module):
    def __init__(self, hidden_dim=256, out_dim=128):
        super().__init__()
        self.conv1 = HeteroConv({
            ('chunk', 'mentions', 'entity'): SAGEConv((-1, -1), hidden_dim),
            ('entity', 'mentioned_in', 'chunk'): SAGEConv((-1, -1), hidden_dim),
            ('entity', 'related_to', 'entity'): SAGEConv((-1, -1), hidden_dim),
            ('entity', 'related_to_rev', 'entity'): SAGEConv((-1, -1), hidden_dim),
        }, aggr='mean')
        self.conv2 = HeteroConv({
            ('chunk', 'mentions', 'entity'): SAGEConv((-1, -1), out_dim),
            ('entity', 'mentioned_in', 'chunk'): SAGEConv((-1, -1), out_dim),
            ('entity', 'related_to', 'entity'): SAGEConv((-1, -1), out_dim),
            ('entity', 'related_to_rev', 'entity'): SAGEConv((-1, -1), out_dim),
        }, aggr='mean')

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        x_dict = {k: F.normalize(v, p=2, dim=-1) for k, v in x_dict.items()}
        return x_dict

def edge_scores(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, edges: torch.Tensor) -> torch.Tensor:
    c = chunk_emb[edges[:, 0]]
    e = entity_emb[edges[:, 1]]
    return (c * e).sum(dim=-1)

@torch.no_grad()
def evaluate_retrieval(model: nn.Module, data: HeteroData, eval_edges: torch.Tensor, k_list=(5, 10, 20)):
    model.eval()
    z = model(data.x_dict, data.edge_index_dict)
    chunk_z = z['chunk']
    entity_z = z['entity']

    recalls = {k: [] for k in k_list}
    mrr = []

    for c_idx, e_idx in eval_edges.tolist():
        scores = torch.matmul(entity_z, chunk_z[c_idx])
        ranking = torch.argsort(scores, descending=True)
        rank_pos = (ranking == e_idx).nonzero(as_tuple=False)
        if len(rank_pos) == 0:
            continue
        rank = int(rank_pos[0].item()) + 1

        for k in k_list:
            recalls[k].append(1.0 if rank <= k else 0.0)
        mrr.append(1.0 / rank)

    metrics = {f'Recall@{k}': float(np.mean(v)) if v else 0.0 for k, v in recalls.items()}
    metrics['MRR'] = float(np.mean(mrr)) if mrr else 0.0
    return metrics


In [ ]:
model = RetrievalHeteroSAGE(hidden_dim=256, out_dim=128).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

data = data.to(DEVICE)
train_edges = train_edges.to(DEVICE)
val_edges = val_edges.to(DEVICE)
test_edges = test_edges.to(DEVICE)

epochs = 30
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()

    z = model(data.x_dict, data.edge_index_dict)
    pos_scores = edge_scores(z['chunk'], z['entity'], train_edges)
    neg_edges = sample_negative(train_edges.cpu()).to(DEVICE)
    neg_scores = edge_scores(z['chunk'], z['entity'], neg_edges)

    logits = torch.cat([pos_scores, neg_scores], dim=0)
    labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)], dim=0)
    loss = F.binary_cross_entropy_with_logits(logits, labels)

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0 or epoch == 1:
        val_metrics = evaluate_retrieval(model, data, val_edges)
        print(f'Epoch {epoch:03d} | loss={loss.item():.4f} | val={val_metrics}')

test_metrics = evaluate_retrieval(model, data, test_edges)
print('Test metrics:', test_metrics)


In [ ]:
# Инференс эмбеддингов и запись обратно в Neo4j
model.eval()
with torch.no_grad():
    z = model(data.x_dict, data.edge_index_dict)

chunk_emb = z['chunk'].detach().cpu().numpy()
entity_emb = z['entity'].detach().cpu().numpy()

chunk_records = [
    {'chunk_id': cid, 'gnn_embedding': emb.tolist()}
    for cid, emb in zip(chunks_df['chunk_id'].tolist(), chunk_emb)
]
entity_records = [
    {'entity_id': eid, 'gnn_embedding': emb.tolist()}
    for eid, emb in zip(entities_df['entity_id'].tolist(), entity_emb)
]

with driver.session() as session:
    session.run("""
    UNWIND $rows AS row
    MATCH (c:Chunk {chunk_id: row.chunk_id})
    SET c.gnn_embedding = row.gnn_embedding,
        c.gnn_model_version = $model_version,
        c.gnn_updated_at = datetime()
    """, rows=chunk_records, model_version='graphsage_v1')

    session.run("""
    UNWIND $rows AS row
    MATCH (e:Entity {entity_id: row.entity_id})
    SET e.gnn_embedding = row.gnn_embedding,
        e.gnn_model_version = $model_version,
        e.gnn_updated_at = datetime()
    """, rows=entity_records, model_version='graphsage_v1')

print('Embeddings upserted:', len(chunk_records), 'chunks,', len(entity_records), 'entities')


In [ ]:
# Demo retrieval: top-N entities для произвольного chunk
sample_chunk_idx = 0
sample_chunk_id = chunks_df.iloc[sample_chunk_idx]['chunk_id']
sample_text = chunks_df.iloc[sample_chunk_idx]['text'][:220]

with torch.no_grad():
    z = model(data.x_dict, data.edge_index_dict)
    scores = torch.matmul(z['entity'], z['chunk'][sample_chunk_idx])
    topk = torch.topk(scores, k=min(10, len(scores))).indices.cpu().tolist()

print('Chunk:', sample_chunk_id)
print('Text preview:', sample_text)
print('Top entities:')
for i, idx in enumerate(topk, 1):
    row = entities_df.iloc[idx]
    print(f'{i:02d}. {row.entity_id} | {row.entity_text}')


In [ ]:
driver.close()
